My solution for the titanic ml challenge

First I load training data from csv to data frame

In [15]:
import pandas as pd

df = pd.read_csv("./titanic/train.csv")

#df.shape
display(df.head(5))

#print("Survived females: ", ((df["Sex"] == "female") & (df["Survived"] == 1)).sum())
#print("Survived males: ", ((df["Sex"] == "male") & (df["Survived"] == 1)).sum())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Some data exploration, seeing how classes affected survivability and who exactly survived

In [16]:
data = []
for pclass in sorted(df["Pclass"].unique()):
    total = (df["Pclass"] == pclass).sum()
    total_males = ((df["Sex"] == "male") & (df["Pclass"] == pclass)).sum()
    total_females = ((df["Sex"] == "female") & (df["Pclass"] == pclass)).sum()

    survived = ((df["Pclass"] == pclass) & (df["Survived"] == 1))
    men_survived = ((df["Sex"] == "male") & survived).sum()
    female_survived = ((df["Sex"] == "female") & survived).sum()

    kids = ((df["Age"] < 18) & survived).sum()

    data.append({
        "Class": pclass,
        "Passengers": total,
        "Survived": survived.sum(),
        "Survived children": kids, 
        "Percentage (%)": survived.sum() / total * 100,
        "Survived men": men_survived,
        "Total men": total_males,
        "Survived females": female_survived,
        "Total Females": total_females
    })

summary = pd.DataFrame(data)
display(summary.style
        .format({"Percentage (%)": "{:.2f}%"})
        .hide(axis="index"))

Class,Passengers,Survived,Survived children,Percentage (%),Survived men,Total men,Survived females,Total Females
1,216,136,11,62.96%,45,122,91,94
2,184,87,21,47.28%,17,108,70,76
3,491,119,29,24.24%,47,347,72,144


Missing values

In [17]:
#df.shape

df.info()

print(df["Cabin"].isna().sum())
print(df["Age"].isna().sum())
print(df["Embarked"].isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
687
177
2


Missing values:
- cabin (687)
- age (177)
- Embarked (2)

I will skip Cabin column entirely for now, in future i will try to make it known/unknown and see if model improves

For age i will imputate the missing values with median age and also test model with age - known/unknown feature

Embarked - I will just fill this with the most popular port of embarkation present in dataset

Data will be filled based on training groups

Splitting data into training and testing split

In [18]:
from typing import cast

from pandas import DataFrame, Series
from sklearn.model_selection import train_test_split


X = df.drop(columns=["Cabin", "Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = cast(
    tuple[DataFrame, DataFrame, Series, Series],
    train_test_split(X, y, test_size=0.2, random_state=42, stratify=y),
)

print(X_train.shape)
print(X_test.shape)

(712, 10)
(179, 10)


Baseline - dummy classifier with most_frequent strategy so it will just spit the most frequent class in our dataset as its prediction. This is my baseline model which i will compare to my Logistic Regression

In [19]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(X_train, y_train)

print("Classes:", dummy.classes_)
print("Class distribution:")
print(y_train.value_counts(normalize=True))

y_train_pred = dummy.predict(X_train)

accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred, zero_division=0)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)

print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall : ", recall)
print("F1: ", f1)

Classes: [0 1]
Class distribution:
Survived
0    0.616573
1    0.383427
Name: proportion, dtype: float64
Accuracy:  0.6165730337078652
Precision:  0.0
Recall :  0.0
F1:  0.0


Cross validation to see how our dummy performs on different data

In [20]:
from sklearn.model_selection import cross_val_score


scores = cross_val_score(dummy, X_train, y_train, cv=5, scoring="accuracy")
print("CV accuracy scores: ", scores)
print("CV mean accuracy: ", scores.mean())

CV accuracy scores:  [0.61538462 0.61538462 0.61267606 0.61971831 0.61971831]
CV mean accuracy:  0.6165763813651137


Now I want to classify my features into numeric and categorical features because I want to preprocess them first to fill blank values 

In [22]:
numeric_features = ["Age", "Fare", "SibSp", "Parch"]

categorical_features = ["Pclass", "Sex", "Embarked"]

features = numeric_features + categorical_features

X_train_selected = X_train[features]

#X_train_selected.shape
X_train_selected.head(5)

,Age,Fare,SibSp,Parch,Pclass,Sex,Embarked
692,NaN,56.4958,0,0,3,male,S
481,NaN,0.0000,0,0,2,male,S
527,NaN,221.7792,0,0,1,male,S
855,18.0,9.3500,0,1,3,female,S
801,31.0,26.2500,1,1,2,female,S


Now i want to imputate my numeric features

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
#
#age_imputer = SimpleImputer(strategy="median")
#
#age_imputer.fit(X_train_selected[["Age"]])
#
##age_imputer.statistics_ # Median value of the age
#
#ages_transformed = age_imputer.transform(X_train_selected[["Age"]])
#
#print(ages_transformed)



numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

[[28.5 ]
 [28.5 ]
 [28.5 ]
 [18.  ]
 [31.  ]
 [21.  ]
 [26.  ]
 [28.5 ]
 [28.5 ]
 [31.  ]
 [56.  ]
 [21.  ]
 [28.5 ]
 [56.  ]
 [21.  ]
 [33.  ]
 [21.  ]
 [28.5 ]
 [43.  ]
 [28.5 ]
 [30.  ]
 [32.  ]
 [28.5 ]
 [19.  ]
 [24.  ]
 [45.  ]
 [23.  ]
 [42.  ]
 [39.  ]
 [16.  ]
 [28.5 ]
 [66.  ]
 [28.  ]
 [51.  ]
 [47.  ]
 [18.  ]
 [36.  ]
 [30.  ]
 [22.  ]
 [28.5 ]
 [25.  ]
 [28.5 ]
 [32.  ]
 [28.5 ]
 [19.  ]
 [52.  ]
 [71.  ]
 [32.  ]
 [19.  ]
 [28.5 ]
 [22.  ]
 [50.  ]
 [ 4.  ]
 [36.  ]
 [19.  ]
 [30.  ]
 [28.5 ]
 [30.  ]
 [21.  ]
 [30.  ]
 [18.  ]
 [19.  ]
 [38.  ]
 [10.  ]
 [28.5 ]
 [29.  ]
 [28.5 ]
 [28.  ]
 [39.  ]
 [ 2.  ]
 [30.  ]
 [25.  ]
 [14.  ]
 [35.  ]
 [17.  ]
 [28.5 ]
 [18.  ]
 [23.  ]
 [50.  ]
 [21.  ]
 [58.  ]
 [ 3.  ]
 [13.  ]
 [57.  ]
 [16.  ]
 [28.5 ]
 [60.  ]
 [22.  ]
 [30.  ]
 [17.  ]
 [25.  ]
 [27.  ]
 [36.  ]
 [28.5 ]
 [28.5 ]
 [29.  ]
 [30.  ]
 [28.5 ]
 [51.  ]
 [16.  ]
 [28.5 ]
 [ 1.  ]
 [22.  ]
 [28.5 ]
 [ 7.  ]
 [28.5 ]
 [28.5 ]
 [33.  ]
 [ 7.  ]
 [32.  ]
 [50.  ]
 